# Prompt Optimizer v2 - QLoRA Fine-Tuning

Fine-tune Qwen2.5-3B-Instruct for automatic prompt optimization.

**Setup:**
1. Change runtime type to **T4 GPU** (Runtime → Change runtime type)
2. Run all cells — dataset downloads automatically from GitHub

**Dataset v2:** 1081 items, 8 categories, 100% unique outputs

**Changes from v1:**
- 37% more training data (863 → 1081 total)
- 5 epochs (was 3)
- Anti-prefix system prompt to prevent model adding "Optimized Prompt:"

**Expected training time:** ~3-4 hours on T4 GPU

In [ ]:
#@title 1. Install Dependencies
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Change runtime type to T4 GPU.')
    raise RuntimeError('No GPU available. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 2. Configuration

# Model settings
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct" #@param ["unsloth/Qwen2.5-3B-Instruct", "unsloth/Qwen2.5-1.5B-Instruct", "unsloth/Qwen2.5-7B-Instruct"]

# LoRA settings
LORA_R = 16 #@param {type:"integer"}
LORA_ALPHA = 16 #@param {type:"integer"}
LORA_DROPOUT = 0 #@param {type:"number"}
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training settings
BATCH_SIZE = 4 #@param {type:"integer"}
GRADIENT_ACCUMULATION = 4 #@param {type:"integer"}
LEARNING_RATE = 2e-4 #@param {type:"number"}
NUM_EPOCHS = 5 #@param {type:"integer"}
MAX_SEQ_LENGTH = 2048 #@param {type:"integer"}
WARMUP_RATIO = 0.1 #@param {type:"number"}
WEIGHT_DECAY = 0.01 #@param {type:"number"}

# Paths
OUTPUT_DIR = "./prompt-optimizer-model"

print(f"Model: {MODEL_NAME}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Training: {NUM_EPOCHS} epochs, batch={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
#@title 3. Load Model and Tokenizer

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
#@title 4. Load and Format Dataset
import json
from datasets import Dataset

def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Load data - clone from GitHub if not present
import os

if not os.path.exists('train.jsonl'):
    print('Cloning dataset from GitHub...')
    !git clone --depth 1 https://github.com/Petsku01/prompt-optimizer.git /content/prompt-optimizer
    !cp /content/prompt-optimizer/data/train.jsonl .
    !cp /content/prompt-optimizer/data/val.jsonl .

train_data = load_jsonl('train.jsonl')
val_data = load_jsonl('val.jsonl')

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
if train_data:
    print(f"\nExample:")
    print(f"  Input: {train_data[0]['input']}")
    print(f"  Output: {train_data[0]['output'][:100]}...")

# System prompt - ANTI-PREFIX to prevent model adding "Optimized Prompt:"
SYSTEM_PROMPT = "You are a prompt engineering expert. Transform vague, underspecified prompts into clear, specific, and effective ones. Preserve the original intent while adding concrete details, structure, and actionable guidance. IMPORTANT: Output ONLY the optimized prompt text. Do NOT add any prefix like 'Optimized Prompt:' or 'Here is the optimized prompt:'. Start directly with the optimized content."

def format_example(example):
    """Format a single example into the Qwen2.5 chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{example['instruction']}\n\nPrompt to optimize: {example['input']}"},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = Dataset.from_list([format_example(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_example(ex) for ex in val_data]) if val_data else None

print(f"\nFormatted dataset ready!")
print(f"Train: {len(train_dataset)} examples")
if val_dataset:
    print(f"Val: {len(val_dataset)} examples")

In [ ]:
#@title 5. Train!

from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps" if val_dataset else "no",
    eval_steps=100 if val_dataset else None,
    bf16=False,
    fp16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",  # Change to "wandb" if you want W&B logging
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,  # Pack multiple short examples into one sequence
)

# Show memory usage
gpu_stats = torch.cuda.get_device_properties(0)
start_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"VRAM reserved: {start_memory} GB")
print(f"\nStarting training...")

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps: {trainer_stats.global_step}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Peak memory: {round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)} GB")

In [ ]:
#@title 6. Test the Model

from unsloth import FastLanguageModel

# Switch to inference mode
FastLanguageModel.for_inference(model)

test_prompts = [
    "write about dogs",
    "fix my python code",
    "analyze sales data",
    "translate to Finnish",
    "give me ideas for a mobile app",
    "how to set up docker",
    "summarize this article",
    "improve this email",
    "act as a career counselor",
    "what is Kubernetes",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Optimize the following prompt to be clear, specific, and effective. Preserve the original intent while adding structure, context, and constraints where appropriate.\n\nPrompt to optimize: {prompt}"},
    ]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"VAGUE: {prompt}")
    print(f"OPTIMIZED: {response.strip()}")

In [ ]:
#@title 7. Save Model

# Save LoRA adapters locally
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"LoRA adapters saved to {OUTPUT_DIR}/lora")

# Optional: Merge and save full model (larger file)
# model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer)

# Optional: Push to HuggingFace Hub
# model.push_to_hub_merged("your-username/prompt-optimizer-qwen2.5-3b", tokenizer)
# Or push just the LoRA adapters:
# model.push_to_hub(f"your-username/prompt-optimizer-qwen2.5-3b-lora")

print("\nModel saved! To use locally:")
print("""from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained("./prompt-optimizer-model/lora")
FastLanguageModel.for_inference(model)""")

In [ ]:
#@title 8. Download Results

import shutil

# Zip the model for download
shutil.make_archive("prompt-optimizer-lora", "zip", f"{OUTPUT_DIR}/lora")
print(f"Model zipped: prompt-optimizer-lora.zip")
print("Download it from the Colab file browser (folder icon on the left)")

# Also save training config for reproducibility
config = {
    "model_name": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "train_examples": len(train_data),
    "val_examples": len(val_data),
}
with open("training_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("\nTraining config saved to training_config.json")